# Step 1 causal-anchor FID

This notebook evaluates the **motion quality of Step 1 anchors without invoking Step 2**. For every validation clip, the planner is rolled out causally. Its predicted anchors replace the corresponding GT causal-codec anchors, while all non-anchor tokens remain GT. The complete token sequence is decoded with the causal body codecs and passed to the existing motion-FID evaluator.

Two references are reported:

1. **Causal codec reconstruction** — the primary anchor-only comparison. This measures the distribution shift introduced by replacing GT anchors.
2. **Canonicalized raw motion** — the absolute reconstruction comparison, including codec error.

> This is an oracle-gap anchor diagnostic. It is not end-to-end Step 1 + Step 2 FID because the non-anchor tokens remain GT.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'motion_generation').is_dir() and (candidate / 'evaluation').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find repository root above {start}')

PROJECT_ROOT = find_project_root(Path.cwd())
print('project:', PROJECT_ROOT)

## Configuration

Start with 128 clips to verify the pipeline. Set `MAX_CLIPS = 0` for the final all-635 validation result. The subset is deterministic, so every condition uses identical clips. `self_forcing_best_rollout` is the current candidate; `q0_6k_final` is the earlier causal multipart baseline.

In [ ]:
CHECKPOINTS = {
    'self_forcing_best_rollout': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_self_forcing_q0q3_full/best_rollout',
    'q0_6k_final': PROJECT_ROOT / 'checkpoints/step1_multipart_fixed_gap3_main6000/final',
}
CODEC_ROOT = PROJECT_ROOT / 'checkpoints/causal_multipart_rvqvae'
CAUSAL_CODECS = {
    part: CODEC_ROOT / f'causal_rvq_{part}_512x4_scratch/model/best.pth'
    for part in ('upper', 'lower', 'feet', 'hands')
}
OUTPUT_DIR = PROJECT_ROOT / 'motion_generation/outputs/step1_anchor_fid'

DEVICE = 'cuda:0'
MAX_CLIPS = 128       # 0 = all 635 validation clips
SUBSET_SEED = 42
ROLLOUT_BATCH_SIZE = 8
FID_BATCH_SIZE = 64
NUM_WORKERS = 0
CANONICALIZE_RAW_ROOT = True

RUN_EVALUATION = True
EXPORT_ONLY = False
FID_ONLY = False      # reuse an already completed export
assert not (EXPORT_ONLY and FID_ONLY)

In [ ]:
required = {
    **{f'planner:{label}': path for label, path in CHECKPOINTS.items()},
    **{f'codec:{part}': path for part, path in CAUSAL_CODECS.items()},
    'evaluator_checkpoint': PROJECT_ROOT / 'checkpoints/eval_model/best_model.pt',
    'evaluator_config': PROJECT_ROOT / 'evaluation/config/train_bert_orig.yaml',
    'evaluator_mean': PROJECT_ROOT / 'evaluation/stats/humanml3d/guoh3dfeats/mean.pt',
    'evaluator_std': PROJECT_ROOT / 'evaluation/stats/humanml3d/guoh3dfeats/std.pt',
}
preflight = pd.DataFrame([
    {'item': label, 'path': str(path), 'exists': path.exists()}
    for label, path in required.items()
])
display(preflight)
missing = preflight.loc[~preflight['exists'], 'item'].tolist()
assert not missing, f'Missing prerequisites: {missing}'

## Export substituted motions and compute FID

The script generates actual causal rollouts, verifies that each rollout target matches the dense causal-token export, substitutes anchors, decodes every condition, and computes FID/diversity. It also exports decoded RMSE and dynamics diagnostics.

In [ ]:
command = [
    sys.executable,
    str(PROJECT_ROOT / 'motion_generation/scripts/evaluate_step1_anchor_fid.py'),
    '--output_dir', str(OUTPUT_DIR),
    '--device', DEVICE,
    '--max_clips', str(MAX_CLIPS),
    '--subset_seed', str(SUBSET_SEED),
    '--rollout_batch_size', str(ROLLOUT_BATCH_SIZE),
    '--fid_batch_size', str(FID_BATCH_SIZE),
    '--num_workers', str(NUM_WORKERS),
]
for label, path in CHECKPOINTS.items():
    command.extend(['--checkpoint', f'{label}={path}'])
for part, path in CAUSAL_CODECS.items():
    command.extend([f'--{part}_ckpt', str(path)])
if not CANONICALIZE_RAW_ROOT:
    command.append('--no_canonicalize_raw_root')
if EXPORT_ONLY:
    command.append('--export_only')
if FID_ONLY:
    command.append('--fid_only')

print(' '.join(map(str, command)))
if RUN_EVALUATION:
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
else:
    print('Evaluation skipped; existing outputs will be read below.')

## Primary result: FID relative to causal codec reconstruction

This is the requested comparison. Lower is better. The causal reconstruction compared with itself should be approximately zero. The planner's score is the additional distribution shift caused by its anchors.

In [ ]:
fid = pd.read_csv(OUTPUT_DIR / 'anchor_fid_metrics.csv')
codec_relative = (
    fid[fid['reference'] == 'causal_codec_reconstruction']
    .sort_values('FID_norm_by_GT')
    .reset_index(drop=True)
)
display(codec_relative[[
    'condition', 'num_clips', 'FID_norm_by_GT', 'FID_raw',
    'delta_fid_norm_vs_codec_floor', 'Diversity_GT', 'Diversity_Gen',
    'diversity_gap',
]].round(6))

## Secondary result: FID relative to raw motion

This table includes the causal codec's reconstruction cost. Compare each planner with `causal_codec_reconstruction`; `delta_fid_norm_vs_codec_floor` is the extra cost introduced by substituted anchors. Do not compare this table with historical non-canonicalized-root numbers unless the same protocol is rerun.

In [ ]:
raw_reference = fid[fid['reference'].str.startswith('raw_motion_')].copy()
raw_reference = raw_reference.sort_values('FID_norm_by_GT').reset_index(drop=True)
display(raw_reference[[
    'reference', 'condition', 'num_clips', 'FID_norm_by_GT', 'FID_raw',
    'delta_fid_norm_vs_codec_floor', 'Diversity_GT', 'Diversity_Gen',
]].round(6))

## Supporting diagnostics

FID is distribution-level and unpaired. These decoded errors reveal whether a favorable FID hides large clip-level anchor damage. `oracle_previous_gt_anchor_copy` uses the preceding GT anchor and is not deployable; `seed_hold` is the deployable persistence control.

In [ ]:
rollout_tokens = pd.read_csv(OUTPUT_DIR / 'rollout_token_summary.csv')
decoded = pd.read_csv(OUTPUT_DIR / 'decoded_metrics_summary.csv')
manifest = pd.read_csv(OUTPUT_DIR / 'anchor_fid_manifest.csv')
display(rollout_tokens.round(6))
display(decoded.sort_values('codec_relative_rmse').round(6))
display(manifest[[
    'token_frames', 'motion_frames', 'predicted_anchor_count',
    'predicted_anchor_fraction',
]].describe().round(4))

## Interpretation gate

Prefer the new planner only when all of the following broadly agree:

- codec-relative anchor FID improves over `q0_6k_final`;
- diversity stays close to the causal reconstruction rather than collapsing;
- decoded RMSE/velocity/acceleration do not contradict the FID ranking;
- results hold on the all-635 validation run (`MAX_CLIPS = 0`).

A planner can pass this anchor-quality gate even if Step 2 later fails to use the anchors. End-to-end FID remains a separate test after the planner and infiller share a compatible causal token contract.